In [ ]:
# =========================================
# CELL 1 — LOAD RANDOM FOREST MODEL
# =========================================

import os
import joblib

# =========================
# CONFIG
# =========================

MODEL_TYPE = "random_forest"
BASE_DIR = "exported_models"

model_path = os.path.join(BASE_DIR, MODEL_TYPE, "model.pkl")
features_path = os.path.join(BASE_DIR, MODEL_TYPE, "features.pkl")

# =========================
# VALIDATION
# =========================

if not os.path.exists(model_path):
    raise Exception(f"❌ Model not found: {model_path}")

if not os.path.exists(features_path):
    raise Exception(f"❌ Features not found: {features_path}")

# =========================
# LOAD
# =========================

model_rf = joblib.load(model_path)
features_rf = joblib.load(features_path)

# =========================
# INFO
# =========================

print("✅ Random Forest Loaded")
print("Total Features:", len(features_rf))

print("\nSample Features:")
for f in features_rf[:10]:
    print("-", f)

print("\nClasses:", model_rf.classes_)

✅ Random Forest Loaded
Total Features: 18

Sample Features:
- ear_mean
- ear_std
- mar_mean
- mar_std
- motion_mean
- motion_std
- blink_rate
- hr_mean
- hr_std
- hr_range

Classes: ['anxiety' 'depression' 'normal' 'stress']


In [ ]:
# =========================================
# CELL 2 — LOAD & CLEAN RAW DATA
# =========================================

import pandas as pd
import numpy as np

# =========================
# PATH (sesuaikan)
# =========================

video_path = "test_data/video.mp4"
sensor_path = "test_data/raw_sensor.csv"
timestamp_path = "test_data/frame_timestamps.csv"

# =========================
# LOAD DATA
# =========================

sensor_df = pd.read_csv(sensor_path)
frame_df = pd.read_csv(timestamp_path)

print("Raw Sensor Shape:", sensor_df.shape)
print("Raw Frame Shape :", frame_df.shape)

# =========================================
# CLEAN SENSOR
# =========================================

def clean_sensor_data(df):

    df = df.copy()

    # ---------- timestamp ----------
    df["timestamp_ms"] = pd.to_numeric(df["timestamp_ms"], errors="coerce")
    df = df.dropna(subset=["timestamp_ms"])

    # sort + deduplicate
    df = df.sort_values("timestamp_ms")
    df = df.drop_duplicates(subset=["timestamp_ms"])

    # ---------- remove leakage ----------
    if "category" in df.columns:
        df = df.drop(columns=["category"])

    # ---------- fix boolean ----------
    for col in ["hr_valid", "spo2_valid"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper() == "TRUE"

    # ---------- keep only relevant ----------
    keep_cols = [
        "timestamp_ms",
        "heart_rate_bpm",
        "hr_valid",
        "gsr_conductance_us"
    ]

    df = df[[c for c in keep_cols if c in df.columns]]

    # ---------- drop invalid HR ----------
    if "hr_valid" in df.columns:
        df = df[df["hr_valid"] == True]

    return df


# =========================================
# CLEAN FRAME TIMESTAMP
# =========================================

def clean_frame_timestamps(df):

    df = df.copy()

    df["timestamp_ms"] = pd.to_numeric(df["timestamp_ms"], errors="coerce")
    df = df.dropna(subset=["timestamp_ms"])

    df = df.sort_values("timestamp_ms")

    # enforce monotonic increase
    df = df[df["timestamp_ms"].diff().fillna(1) >= 0]

    return df


# =========================
# APPLY CLEANING
# =========================

sensor_df = clean_sensor_data(sensor_df)
frame_df = clean_frame_timestamps(frame_df)

# =========================
# BASIC VALIDATION
# =========================

print("\n✅ CLEANED SENSOR:", sensor_df.shape)
print("Timestamp range (sensor):",
      sensor_df["timestamp_ms"].min(),
      "→",
      sensor_df["timestamp_ms"].max())

print("\n✅ CLEANED FRAME:", frame_df.shape)
print("Timestamp range (frame):",
      frame_df["timestamp_ms"].min(),
      "→",
      frame_df["timestamp_ms"].max())

# =========================
# SANITY CHECK
# =========================

print("\n📊 SENSOR STATS:")
print(sensor_df.describe())

print("\n📊 FRAME HEAD:")
display(frame_df.head())

Raw Sensor Shape: (598, 15)
Raw Frame Shape : (2699, 2)

✅ CLEANED SENSOR: (1, 4)
Timestamp range (sensor): 1774530000000.0 → 1774530000000.0

✅ CLEANED FRAME: (2699, 2)
Timestamp range (frame): 1774526503782 → 1774526563782

📊 SENSOR STATS:
       timestamp_ms  heart_rate_bpm  gsr_conductance_us
count  1.000000e+00             1.0              1.0000
mean   1.774530e+12            60.1              4.2208
std             NaN             NaN                 NaN
min    1.774530e+12            60.1              4.2208
25%    1.774530e+12            60.1              4.2208
50%    1.774530e+12            60.1              4.2208
75%    1.774530e+12            60.1              4.2208
max    1.774530e+12            60.1              4.2208

📊 FRAME HEAD:


,frame_idx,timestamp_ms
0,0,1774526503782
1,1,1774526503802
2,2,1774526503823
3,3,1774526503845
4,4,1774526503867


In [ ]:
# =========================================
# FACIAL FEATURE EXTRACTION (ROBUST VERSION)
# =========================================

import cv2
import numpy as np
from scipy.spatial.distance import euclidean
import mediapipe as mp

# =========================
# INIT MEDIAPIPE
# =========================

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# =========================
# LANDMARK INDEX
# =========================

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [13, 14, 78, 308]

# =========================
# FEATURE FUNCTIONS
# =========================

def compute_EAR(lm, idx):
    p1, p2, p3, p4, p5, p6 = [lm[i] for i in idx]
    return (
        euclidean(p2, p6) + euclidean(p3, p5)
    ) / (2.0 * euclidean(p1, p4) + 1e-6)


def compute_MAR(lm):
    top = lm[MOUTH[0]]
    bottom = lm[MOUTH[1]]
    left = lm[MOUTH[2]]
    right = lm[MOUTH[3]]

    return euclidean(top, bottom) / (euclidean(left, right) + 1e-6)


# =========================================
# MAIN FUNCTION
# =========================================

def extract_facial_features_full_video(session, frame_df):

    cap = cv2.VideoCapture(session["video_path"])

    if not cap.isOpened():
        raise Exception("❌ Cannot open video")

    # =========================
    # NORMALIZE TIME
    # =========================
    frame_df = frame_df.copy()

    start_ts = frame_df["timestamp_ms"].iloc[0]
    frame_df["t"] = frame_df["timestamp_ms"] - start_ts

    # =========================
    # LOOP
    # =========================

    records = []

    prev_landmarks = None

    blink_counter = 0
    blink_frames = 0

    frame_idx = 0
    total_frames = len(frame_df)

    while True:

        ret, frame = cap.read()

        if not ret or frame_idx >= total_frames:
            break

        timestamp = frame_df.iloc[frame_idx]["t"]

        # convert
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = face_mesh.process(rgb)

        if results.multi_face_landmarks:

            face_landmarks = results.multi_face_landmarks[0]

            lm = np.array([
                (p.x, p.y)
                for p in face_landmarks.landmark
            ])

            # =========================
            # EAR
            # =========================
            ear_l = compute_EAR(lm, LEFT_EYE)
            ear_r = compute_EAR(lm, RIGHT_EYE)
            ear = (ear_l + ear_r) / 2

            # =========================
            # MAR
            # =========================
            mar = compute_MAR(lm)

            # =========================
            # MOTION
            # =========================
            motion = 0
            if prev_landmarks is not None:
                motion = np.mean(
                    np.linalg.norm(lm - prev_landmarks, axis=1)
                )

            # =========================
            # BLINK STATE MACHINE
            # =========================
            if ear < 0.11:
                blink_frames += 1
            else:
                if blink_frames >= 2:
                    blink_counter += 1
                blink_frames = 0

            prev_landmarks = lm

            # =========================
            # STORE
            # =========================
            records.append({
                "t": timestamp,
                "EAR": ear,
                "MAR": mar,
                "motion": motion,
                "blink_cumulative": blink_counter
            })

        frame_idx += 1

    cap.release()

    facial_df = pd.DataFrame(records)

    return facial_df

In [ ]:
# =========================================
# CELL 3 — SESSION + FACIAL EXTRACTION
# =========================================

# DEFINE SESSION (WAJIB)
session = {
    "video_path": video_path
}

# RUN EXTRACTION
facial_df = extract_facial_features_full_video(session, frame_df)

# VALIDATION
if facial_df is None or len(facial_df) == 0:
    raise Exception("❌ Facial extraction gagal")

print("✅ Facial extracted")
print("Shape:", facial_df.shape)

print("\n📊 Stats:")
print(facial_df.describe())

print("\nBlink max:", facial_df["blink_cumulative"].max())
print("Motion mean:", facial_df["motion"].mean())

✅ Facial extracted
Shape: (2699, 5)

📊 Stats:
                  t          EAR          MAR       motion  blink_cumulative
count   2699.000000  2699.000000  2699.000000  2699.000000       2699.000000
mean   30009.889589     0.156277     0.007382     0.000777          8.977770
std    17344.358969     0.018258     0.002427     0.000451          6.805915
min        0.000000     0.026653     0.000232     0.000000          0.000000
25%    14970.000000     0.155953     0.005933     0.000516          2.000000
50%    30031.000000     0.160269     0.007373     0.000668          8.000000
75%    45027.500000     0.164083     0.008863     0.000887         15.000000
max    60000.000000     0.176104     0.050855     0.005194         22.000000

Blink max: 22
Motion mean: 0.0007767120796961759


In [ ]:
facial_df = extract_facial_features_full_video(session, frame_df)

In [ ]:
facial_df.shape

(2699, 5)

In [ ]:
facial_df.describe()

,t,EAR,MAR,motion,blink_cumulative
count,2699.000000,2699.000000,2699.000000,2699.000000,2699.000000
mean,30009.889589,0.156278,0.007381,0.000775,9.910337
std,17344.358969,0.018257,0.002405,0.000445,6.898886
min,0.000000,0.027361,0.000218,0.000000,0.000000
25%,14970.000000,0.155901,0.005921,0.000517,3.000000
50%,30031.000000,0.160280,0.007361,0.000667,9.000000
75%,45027.500000,0.164140,0.008861,0.000883,16.000000
max,60000.000000,0.176558,0.048133,0.004662,23.000000


In [ ]:
print("\nBlink max:", facial_df["blink_cumulative"].max())
print("Motion mean:", facial_df["motion"].mean())


Blink max: 23
Motion mean: 0.0007748240978880534


In [ ]:
# =========================================
# CELL 4 — WINDOWING (FACIAL ONLY)
# =========================================

WINDOW_SIZE = 30  # detik
STRIDE = 10       # detik

windows = []

duration_sec = facial_df["t"].max() / 1000

starts = np.arange(0, duration_sec - WINDOW_SIZE + 1, STRIDE)

for start in starts:
    end = start + WINDOW_SIZE

    start_ms = int(start * 1000)
    end_ms = int(end * 1000)

    windows.append((start_ms, end_ms))

print("Total windows:", len(windows))

Total windows: 4


In [ ]:
def clean_and_fix_sensor(sensor_df, sampling_rate=10):

    df = sensor_df.copy()

    # =========================
    # FIX TIMESTAMP
    # =========================
    df["timestamp_ms"] = pd.to_numeric(df["timestamp_ms"], errors="coerce")
    df = df.dropna(subset=["timestamp_ms"])

    df = df.sort_values("timestamp_ms")

    # DETECT COLLAPSE
    if df["timestamp_ms"].nunique() <= 2:
        print("⚠️ Timestamp rusak → reconstruct timeline")

        start = df["timestamp_ms"].iloc[0]
        interval = 1000 / sampling_rate

        df["timestamp_ms"] = [
            start + i * interval for i in range(len(df))
        ]

    # =========================
    # REMOVE LEAKAGE
    # =========================
    if "category" in df.columns:
        df = df.drop(columns=["category"])

    # =========================
    # FIX BOOLEAN
    # =========================
    for col in ["hr_valid", "spo2_valid"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper() == "TRUE"

    # =========================
    # KEEP RELEVANT
    # =========================
    keep_cols = [
        "timestamp_ms",
        "heart_rate_bpm",
        "hr_valid",
        "gsr_conductance_us"
    ]

    df = df[[c for c in keep_cols if c in df.columns]]

    # =========================
    # FILTER VALID
    # =========================
    if "hr_valid" in df.columns:
        df = df[df["hr_valid"] == True]

    return df

In [ ]:
def extract_facial_window_features(facial_df, start_ms, end_ms):

    window = facial_df[
        (facial_df["t"] >= start_ms) &
        (facial_df["t"] <= end_ms)
    ]

    if len(window) < 5:
        return None

    # blink rate
    blink_count = (
        window["blink_cumulative"].iloc[-1] -
        window["blink_cumulative"].iloc[0]
    )

    duration_sec = (end_ms - start_ms) / 1000
    blink_rate = blink_count / duration_sec if duration_sec > 0 else 0

    return {
        "ear_mean": window["EAR"].mean(),
        "ear_std": window["EAR"].std(),
        "mar_mean": window["MAR"].mean(),
        "mar_std": window["MAR"].std(),
        "motion_mean": window["motion"].mean(),
        "motion_std": window["motion"].std(),
        "blink_rate": blink_rate
    }

In [ ]:
sensor_df = pd.read_csv("test_data/raw_sensor.csv")

sensor_df = clean_and_fix_sensor(sensor_df)

print("Sensor shape:", sensor_df.shape)
print(sensor_df.head())
print(sensor_df.describe())

⚠️ Timestamp rusak → reconstruct timeline
Sensor shape: (598, 4)
     timestamp_ms  heart_rate_bpm  hr_valid  gsr_conductance_us
0    1.774530e+12            60.1      True              4.2208
395  1.774530e+12            90.6      True              4.5752
396  1.774530e+12            90.1      True              4.6645
397  1.774530e+12            90.1      True              4.6645
398  1.774530e+12            90.1      True              4.6645
       timestamp_ms  heart_rate_bpm  gsr_conductance_us
count  5.980000e+02      598.000000          598.000000
mean   1.774530e+12       89.401505            4.529423
std    1.727720e+04       13.890642            0.156021
min    1.774530e+12       56.900000            4.132900
25%    1.774530e+12       79.300000            4.486100
50%    1.774530e+12       85.600000            4.575200
75%    1.774530e+12       96.700000            4.664500
max    1.774530e+12      118.600000            4.754100


In [ ]:
# =========================================
# FIX — EDA & SCR FUNCTIONS
# =========================================

import neurokit2 as nk
import numpy as np

# =========================
# EDA DECOMPOSITION
# =========================

def eda_decomposition(gsr_signal):

    signals, info = nk.eda_process(
        gsr_signal,
        sampling_rate=10  # sesuai pipeline lo
    )

    tonic = signals["EDA_Tonic"].values
    phasic = signals["EDA_Phasic"].values

    return tonic, phasic


# =========================
# SCR FEATURES
# =========================

def extract_scr_features(phasic_signal):

    signals, info = nk.eda_peaks(
        phasic_signal,
        sampling_rate=10
    )

    scr_peaks = signals["SCR_Peaks"]
    scr_amp = signals["SCR_Amplitude"]

    peak_idx = np.where(scr_peaks == 1)[0]

    scr_count = len(peak_idx)

    if scr_count > 0:
        amp_values = scr_amp[peak_idx]
        scr_amp_mean = np.mean(amp_values)
        scr_amp_max = np.max(amp_values)
    else:
        scr_amp_mean = 0
        scr_amp_max = 0

    duration = len(phasic_signal) / 10
    scr_freq = scr_count / duration if duration > 0 else 0

    return {
        "scr_count": scr_count,
        "scr_amp_mean": scr_amp_mean,
        "scr_amp_max": scr_amp_max,
        "scr_frequency": scr_freq
    }

In [ ]:
def extract_sensor_window_features(sensor_df, start_ms, end_ms):

    df = sensor_df.copy()

    # normalize time
    start = df["timestamp_ms"].iloc[0]
    df["t"] = df["timestamp_ms"] - start

    window = df[
        (df["t"] >= start_ms) &
        (df["t"] <= end_ms)
    ]

    if len(window) < 5:
        return None

    window = window.interpolate()

    hr = window["heart_rate_bpm"].values
    gsr = window["gsr_conductance_us"].values

    if len(hr) < 5 or len(gsr) < 5:
        return None

    # =========================
    # HR FEATURES
    # =========================
    hr_mean = np.mean(hr)
    hr_std = np.std(hr)
    hr_range = np.max(hr) - np.min(hr)

    x = np.arange(len(hr))
    hr_slope = np.polyfit(x, hr, 1)[0]

    hr_var_ratio = hr_std / hr_mean if hr_mean > 0 else 0

    # =========================
    # EDA (pakai pipeline lo)
    # =========================
    tonic, phasic = eda_decomposition(gsr)
    scr = extract_scr_features(phasic)

    features = {
        "hr_mean": hr_mean,
        "hr_std": hr_std,
        "hr_range": hr_range,
        "hr_slope": hr_slope,
        "hr_var_ratio": hr_var_ratio,
        "scr_count": scr["scr_count"],
        "scr_amp_mean": scr["scr_amp_mean"],
        "scr_amp_max": scr["scr_amp_max"],
        "scr_frequency": scr["scr_frequency"],
        "eda_tonic_mean": np.mean(tonic),
        "eda_phasic_mean": np.mean(phasic)
    }

    return features

In [ ]:
rows = []

for start_ms, end_ms in windows:

    facial_feat = extract_facial_window_features(
        facial_df,
        start_ms,
        end_ms
    )

    if facial_feat is None:
        continue

    sensor_feat = extract_sensor_window_features(
        sensor_df,
        start_ms,
        end_ms
    )

    if sensor_feat is None:
        continue

    feat = {}
    feat.update(facial_feat)
    feat.update(sensor_feat)

    rows.append(feat)

infer_df = pd.DataFrame(rows)

print("Inference shape:", infer_df.shape)
display(infer_df.head())

Inference shape: (4, 18)


,ear_mean,ear_std,mar_mean,mar_std,motion_mean,motion_std,blink_rate,hr_mean,hr_std,hr_range,hr_slope,hr_var_ratio,scr_count,scr_amp_mean,scr_amp_max,scr_frequency,eda_tonic_mean,eda_phasic_mean
0,0.158219,0.013967,0.006963,0.002029,0.000810,0.000542,0.300000,83.730897,7.243442,54.4,0.021545,0.086509,19,0.114844,0.301109,0.631229,4.571163,0.023792
1,0.158346,0.015467,0.007135,0.002586,0.000717,0.000305,0.300000,91.595681,16.331431,61.7,0.124055,0.178299,24,0.092693,0.307213,0.797342,4.543859,0.000846
2,0.155062,0.020396,0.007969,0.002729,0.000739,0.000309,0.366667,94.338206,16.613685,61.7,-0.017909,0.176108,23,0.130962,0.544771,0.764120,4.494507,-0.009334
3,0.154340,0.021540,0.007798,0.002665,0.000740,0.000317,0.466667,95.213423,16.413643,61.7,-0.085505,0.172388,21,0.165299,0.548053,0.704698,4.468374,-0.004759


In [ ]:
# =========================================
# FIX — PREPARE X_rf (WAJIB)
# =========================================

X_rf = infer_df.copy()

# pastikan semua feature ada
for col in features_rf:
    if col not in X_rf.columns:
        X_rf[col] = 0.0

# urutkan sesuai training
X_rf = X_rf[features_rf]

# bersihkan numeric
X_rf = X_rf.replace([np.inf, -np.inf], np.nan)
X_rf = X_rf.fillna(0.0)

print("✅ X_rf ready")
print("Shape:", X_rf.shape)

display(X_rf.head())

✅ X_rf ready
Shape: (4, 18)


,ear_mean,ear_std,mar_mean,mar_std,motion_mean,motion_std,blink_rate,hr_mean,hr_std,hr_range,hr_slope,hr_var_ratio,scr_count,scr_amp_mean,scr_amp_max,scr_frequency,eda_tonic_mean,eda_phasic_mean
0,0.158219,0.013967,0.006963,0.002029,0.000810,0.000542,0.300000,83.730897,7.243442,54.4,0.021545,0.086509,19,0.114844,0.301109,0.631229,4.571163,0.023792
1,0.158346,0.015467,0.007135,0.002586,0.000717,0.000305,0.300000,91.595681,16.331431,61.7,0.124055,0.178299,24,0.092693,0.307213,0.797342,4.543859,0.000846
2,0.155062,0.020396,0.007969,0.002729,0.000739,0.000309,0.366667,94.338206,16.613685,61.7,-0.017909,0.176108,23,0.130962,0.544771,0.764120,4.494507,-0.009334
3,0.154340,0.021540,0.007798,0.002665,0.000740,0.000317,0.466667,95.213423,16.413643,61.7,-0.085505,0.172388,21,0.165299,0.548053,0.704698,4.468374,-0.004759


In [ ]:
# =========================================
# CELL 3 — RF PREDICTION
# =========================================

# =========================
# PREDICT
# =========================

preds_rf = model_rf.predict(X_rf)
probs_rf = model_rf.predict_proba(X_rf)

classes_rf = model_rf.classes_

# =========================
# BUILD RESULT
# =========================

result_rf = pd.DataFrame(probs_rf, columns=classes_rf)
result_rf["prediction"] = preds_rf
result_rf["confidence"] = result_rf[classes_rf].max(axis=1)

# =========================
# OUTPUT
# =========================

print("✅ Prediction per window")
display(result_rf)

✅ Prediction per window


,anxiety,depression,normal,stress,prediction,confidence
0,0.196944,0.248987,0.221099,0.332971,stress,0.332971
1,0.137025,0.352414,0.333977,0.176585,depression,0.352414
2,0.192838,0.310343,0.189793,0.307025,depression,0.310343
3,0.187449,0.354328,0.114282,0.343941,depression,0.354328


In [ ]:
# =========================================
# CELL 4 — FINAL AGGREGATION
# =========================================

# =========================
# AVERAGE PROBABILITY
# =========================

avg_probs_rf = result_rf[classes_rf].mean()

# =========================
# FINAL DECISION
# =========================

final_label_rf = avg_probs_rf.idxmax()
final_conf_rf = avg_probs_rf.max()

# =========================
# OUTPUT
# =========================

print("🎯 FINAL RESULT (Random Forest)")
print("Label:", final_label_rf)
print(f"Confidence: {final_conf_rf*100:.2f}%")

print("\n📊 Full Distribution:")
for c in classes_rf:
    print(f"{c}: {avg_probs_rf[c]*100:.2f}%")

🎯 FINAL RESULT (Random Forest)
Label: depression
Confidence: 31.65%

📊 Full Distribution:
anxiety: 17.86%
depression: 31.65%
normal: 21.48%
stress: 29.01%
